# SEPA Entry Point Analysis

**Goal:** Research and pinpoint entry strategy by analyzing **history of SEPA** (Specific Entry Point Analysis) and **calculating edge**. The framework is Minervini-style: **Trend**, **Base**, and **Pivot** evaluation.

- **Trend:** Uptrend (50 > 150 > 200 SMA, 200d rising), Trend Template score (0–8).
- **Base:** SEPA base types (Cup w/ handle, Low cheat, Power Play, etc.), resistance, depth %, duration.
- **Pivot:** Tight consolidation (3–10 days, range < 8%), optional tight closes; pivot must sit inside a base for a clear buy point.

We will:
1. Load historical daily/weekly data for a universe of symbols.
2. At each historical date, compute trend, base, and pivot state (as-if we didn't see the future).
3. Define entry rules (e.g. breakout above resistance, pivot in base near resistance).
4. Compute forward returns (e.g. 5d, 21d, 63d) from each entry.
5. Aggregate **edge**: win rate, average return, count by rule and base type.

## 1. Imports and config

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Project root: allow importing turtle_rpm (notebooks/ is under project root)
ROOT = Path.cwd() if (Path.cwd() / "turtle_rpm").is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from turtle_rpm.sepa import (
    MIN_DAILY_BARS,
    compute_smas,
    find_bases,
    get_daily_ohlcv,
    pivot_forming,
    pivot_in_base,
    to_weekly,
    uptrend_at_date,
)
from turtle_rpm.leadership import add_52w_high_low, trend_template, rs_ratio_6m

# Symbols to analyze (expand as needed)
SYMBOLS = ["SPY", "QQQ", "AAPL", "MSFT", "NVDA"]
# Forward return windows (trading days)
FORWARD_DAYS = [5, 21, 63]  # ~1w, ~1m, ~3m
# Minimum trend template score for "quality" filter (optional)
MIN_TREND_SCORE = 5

## 2. Load data and build as-of state per date

For each symbol we load 5 years of daily data, then iterate over **weekly** dates (to reduce runtime). At each date we slice data to that date only and compute: uptrend, pivot, bases, pivot-in-base, trend template score. We also detect if that week is a **breakout** (close >= resistance).

In [ ]:
def build_asof_state(symbol: str, years: int = 5) -> pd.DataFrame:
    """
    For each trading day (we sample weekly for speed), compute SEPA state as-of that date.
    Returns DataFrame with columns: date, close, uptrend, pivot_forming, in_base, base_type,
    resistance, distance_pct, tight_closes, trend_score, rs_ratio, is_breakout.
    """
    df_daily = get_daily_ohlcv(symbol, years=years)
    if df_daily.empty or len(df_daily) < MIN_DAILY_BARS:
        return pd.DataFrame()

    daily_smas = compute_smas(df_daily)
    weekly = to_weekly(df_daily)
    if weekly.empty:
        return pd.DataFrame()

    # Sample one date per week (last trading day of week) to keep runtime reasonable
    week_ends = weekly.index.tolist()
    rows = []
    for i, asof_date in enumerate(week_ends):
        # Slice daily and weekly to asof_date (no future data)
        daily_slice = daily_smas[daily_smas.index <= asof_date].copy()
        weekly_slice = weekly[weekly.index <= asof_date].copy()
        if len(daily_slice) < MIN_DAILY_BARS or len(weekly_slice) < 10:
            continue

        pivot = pivot_forming(daily_slice)
        bases = find_bases(weekly_slice, daily_slice, pivot=pivot)
        base_containing = pivot_in_base(pivot, bases) if pivot.get("forming") else None

        close = float(weekly_slice["Close"].iloc[-1])
        resistance = base_containing.get("resistance") if base_containing else None
        is_breakout = (resistance is not None and close >= resistance)

        daily_with_52w = add_52w_high_low(daily_slice)
        rs = rs_ratio_6m(symbol)  # optional: pass asof for true point-in-time; here use latest for speed
        tt = trend_template(daily_with_52w, rs_ratio=rs, at_date=asof_date)

        rows.append({
            "symbol": symbol,
            "date": asof_date,
            "close": close,
            "uptrend": uptrend_at_date(daily_slice, asof_date),
            "pivot_forming": pivot.get("forming", False),
            "in_base": base_containing is not None,
            "base_type": base_containing.get("base_type") if base_containing else None,
            "resistance": resistance,
            "distance_pct": base_containing.get("distance_pct") if base_containing else None,
            "tight_closes": pivot.get("tight_closes", False),
            "trend_score": tt.get("score", 0),
            "rs_ratio": rs,
            "is_breakout": is_breakout,
        })

    return pd.DataFrame(rows)


# Build state for all symbols (can take a few minutes)
all_states = []
for sym in SYMBOLS:
    print(f"Building state for {sym}...")
    state = build_asof_state(sym)
    if not state.empty:
        all_states.append(state)
if all_states:
    states = pd.concat(all_states, ignore_index=True)
    states["date"] = pd.to_datetime(states["date"])
    print(states.shape)
    states.head(10)
else:
    states = pd.DataFrame()
    print("No data.")

## 3. Forward returns

For each row we need the close at `date` and at `date + N` trading days. We'll join with daily closes to compute forward returns.

In [ ]:
def add_forward_returns(states: pd.DataFrame, symbols: list[str], forward_days: list[int]) -> pd.DataFrame:
    """Add columns: ret_5d, ret_21d, ret_63d (or as per forward_days)."""
    out = states.copy()
    for fd in forward_days:
        out[f"ret_{fd}d"] = np.nan

    for sym in symbols:
        df_daily = get_daily_ohlcv(sym, years=5)
        if df_daily.empty:
            continue
        closes = df_daily["Close"]
        idx_sym = out["symbol"] == sym
        for i, row in out[idx_sym].iterrows():
            d = pd.Timestamp(row["date"])
            if d not in closes.index:
                # Nearest prior date
                prior = closes.index[closes.index <= d]
                if len(prior) == 0:
                    continue
                d = prior[-1]
            pos = closes.index.get_loc(d)
            c0 = float(closes.iloc[pos])
            if c0 <= 0:
                continue
            for fd in forward_days:
                pos_f = pos + fd
                if pos_f < len(closes):
                    c_f = float(closes.iloc[pos_f])
                    out.loc[i, f"ret_{fd}d"] = (c_f - c0) / c0

    return out


if not states.empty:
    states = add_forward_returns(states, SYMBOLS, FORWARD_DAYS)
    states.head()

## 4. Entry rules and edge

Define a few entry rule variants, then compute win rate and average return for each rule and forward window.

- **Rule A (Breakout):** `is_breakout` — close >= resistance (pivot in base, breakout that week).
- **Rule B (Near breakout):** `pivot_forming` and `in_base` and `distance_pct <= 3` (within 3% of resistance).
- **Rule C (Trend + base):** `uptrend` and `in_base`, pivot optional.
- **Rule D (Quality breakout):** Breakout + trend_score >= MIN_TREND_SCORE.

In [ ]:
def entry_rule_breakout(row) -> bool:
    return bool(row.get("is_breakout"))


def entry_rule_near_breakout(row) -> bool:
    return (
        bool(row.get("pivot_forming"))
        and bool(row.get("in_base"))
        and row.get("distance_pct") is not None
        and row["distance_pct"] <= 3.0
    )


def entry_rule_trend_base(row) -> bool:
    return bool(row.get("uptrend")) and bool(row.get("in_base"))


def entry_rule_quality_breakout(row) -> bool:
    return bool(row.get("is_breakout")) and (row.get("trend_score") or 0) >= MIN_TREND_SCORE


RULES = {
    "Breakout": entry_rule_breakout,
    "Near breakout (≤3%)": entry_rule_near_breakout,
    "Trend + base": entry_rule_trend_base,
    "Quality breakout (TT≥5)": entry_rule_quality_breakout,
}


def edge_stats(df: pd.DataFrame, ret_col: str) -> dict:
    """Win rate, count, mean return, median return."""
    valid = df[ret_col].dropna()
    if len(valid) == 0:
        return {"count": 0, "win_rate": None, "mean_ret": None, "median_ret": None}
    return {
        "count": len(valid),
        "win_rate": (valid > 0).mean(),
        "mean_ret": valid.mean(),
        "median_ret": valid.median(),
    }


if not states.empty:
    results = []
    for rule_name, rule_fn in RULES.items():
        mask = states.apply(rule_fn, axis=1)
        subset = states.loc[mask]
        for fd in FORWARD_DAYS:
            col = f"ret_{fd}d"
            stats = edge_stats(subset, col)
            stats["rule"] = rule_name
            stats["forward_days"] = fd
            results.append(stats)
    edge_df = pd.DataFrame(results)
    display(edge_df)

## 5. Edge by base type

Among breakout entries, how does edge vary by **base_type** (Cup w/ handle, Low cheat, etc.)?

In [ ]:
if not states.empty and "base_type" in states.columns:
    breakouts = states[states["is_breakout"]].copy()
    breakouts["base_type"] = breakouts["base_type"].fillna("unknown")
    by_base = breakouts.groupby("base_type").agg(
        count=("date", "count"),
        win_rate_21d=(f"ret_{FORWARD_DAYS[1]}d", lambda s: (s.dropna() > 0).mean() if s.notna().any() else None),
        mean_ret_21d=(f"ret_{FORWARD_DAYS[1]}d", "mean"),
    ).reset_index()
    display(by_base)

## 6. Summary and next steps

- Compare **win rate** and **mean return** across entry rules and forward windows to choose a primary entry strategy.
- Use **base_type** breakdown to overweight or filter by base type (e.g. Cup w/ handle vs Low cheat).
- Refine: add **volume at pivot** (below average = better), **tight_closes**, or **trend_score** filters and re-run edge stats.
- Extend symbol universe and time range for more robust statistics; consider walk-forward or out-of-sample period.